Scraping PDF Work flow:
1. Fetch the HTML of the following page: https://www.svsu.edu/matholympics/pastexamsandsolutions/

2. Extract all href="...pdf" links

3. Download the PDFs into a google drive folder

In [5]:
!rm -rf /content/drive
from google.colab import drive
drive.mount('/content/drive')



Mounted at /content/drive


Create Directory where pdfs will live

In [6]:
import os

OUT_DIR = "/content/drive/MyDrive/Math Olympiad Competition/svsu_pdfs"
os.makedirs(OUT_DIR, exist_ok=True)

OUT_DIR


'/content/drive/MyDrive/Math Olympiad Competition/svsu_pdfs'

Install beautiful soup and import necessary libraries

In [2]:
!pip -q install beautifulsoup4 requests

import re
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

In [7]:
#
url = "https://web.evanchen.cc/problems.html"

headers = {
    "User-Agent": "Mozilla/5.0 (compatible; colab-scraper/1.0; +https://colab.research.google.com/)"
}

resp = requests.get(url, headers=headers, timeout=30)
resp.raise_for_status()

soup = BeautifulSoup(resp.text, "html.parser")

pdf_urls = []
for a in soup.select("a[href]"):
    href = a.get("href")
    if not href:
        continue
    abs_url = urljoin(url, href)
    if abs_url.lower().endswith(".pdf"):
        pdf_urls.append(abs_url)

# de-duplicate while preserving order
seen = set()
pdf_urls = [u for u in pdf_urls if not (u in seen or seen.add(u))]

len(pdf_urls), pdf_urls[:5]

(203,
 ['https://web.evanchen.cc/upload/public-CV.pdf',
  'https://web.evanchen.cc/exams/IMO-1997-notes.pdf',
  'https://web.evanchen.cc/exams/IMO-1998-notes.pdf',
  'https://web.evanchen.cc/exams/IMO-1999-notes.pdf',
  'https://web.evanchen.cc/exams/IMO-2000-notes.pdf'])

In [8]:
import os
import time

def safe_filename_from_url(url: str) -> str:
    # use the last URL path segment
    name = url.split("/")[-1].split("?")[0].split("#")[0]
    # basic cleanup
    name = re.sub(r"[^A-Za-z0-9._-]+", "_", name)
    return name or "file.pdf"

def download_pdf(url: str, out_dir: str, sleep_s: float = 0.2) -> str:
    filename = safe_filename_from_url(url)
    out_path = os.path.join(out_dir, filename)

    if os.path.exists(out_path) and os.path.getsize(out_path) > 0:
        return f"SKIP (exists): {filename}"

    r = requests.get(url, headers=headers, stream=True, timeout=60)
    r.raise_for_status()

    # Optional: verify content-type looks like a PDF
    ctype = (r.headers.get("Content-Type") or "").lower()
    if "pdf" not in ctype and not filename.lower().endswith(".pdf"):
        return f"SKIP (not pdf?): {url}"

    with open(out_path, "wb") as f:
        for chunk in r.iter_content(chunk_size=1024 * 256):
            if chunk:
                f.write(chunk)

    time.sleep(sleep_s)  # be polite to the server
    return f"OK: {filename}"

results = [download_pdf(u, OUT_DIR) for u in pdf_urls]
results[:20], sum(r.startswith("OK:") for r in results)


KeyboardInterrupt: 